# EviChange: train the main-cover model (Phase 4b), no Google Drive needed

Runs the 4b fine-tune from start to finish. You upload the data from your computer, and the results download at the end.

**Before you start**
- **Runtime → Change runtime type → A100 GPU.** A T4 or L4 does not have enough memory to train the 7B model.
- On your computer, have these 3 files ready from `D:\EviChange\data\finetune\`:
  `chips.zip`, `cover_train.jsonl`, `cover_holdout.jsonl`.

**Run the cells in order: Runtime → Run all works too.**
Total time is about 1 hour: the model download takes about 1 minute, the baseline about 5 minutes, training 30–45 minutes, and the evaluation about 5 minutes.
Cell 5 downloads `ft_results.zip` at the end. **Send that file to P2.** It holds the trained adapter and the predictions.

If the browser tab is closed or Colab disconnects, everything in `/content` is lost, and you have to run the notebook again from cell 1.

## 1. Install, and check the GPU

In [ ]:
# Colab ships an old torchao that the current peft refuses to load (ImportError: ... torchao 0.10.0).
# We don't quantize, so it isn't needed: remove it first, then install the training libraries.
!pip -q uninstall -y torchao
!pip -q install -U "transformers>=4.57" peft accelerate pillow

import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> A100 GPU, then run this cell again"
name = torch.cuda.get_device_name(0)
mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(name, round(mem), "GB")
assert mem > 35, f"{name} has {mem:.0f} GB: training the 7B model needs an A100 (40 or 80 GB). Change the runtime and start again."

import importlib.util
assert importlib.util.find_spec("torchao") is None, "torchao is still installed: run this cell again"
print("ready")

## 2. Get the code

In [ ]:
!rm -rf /content/EviChange && git clone -q --depth 1 https://github.com/nachikethpt/EviChange /content/EviChange
!ls /content/EviChange/scripts/finetune_train.py

## 3. Upload the data

Click **Choose files** and pick the **3 files** from `D:\EviChange\data\finetune\`. Hold Ctrl to select all three at once:
`chips.zip` (46 MB, the slowest), `cover_train.jsonl` and `cover_holdout.jsonl`.
If you run this cell again later, files that are already uploaded are kept, so you only need to pick the missing ones.

In [ ]:
import os, zipfile
from pathlib import Path
from google.colab import files

FT = Path("/content/ft"); FT.mkdir(exist_ok=True)
os.chdir(FT)
NEEDED = ["chips.zip", "cover_train.jsonl", "cover_holdout.jsonl"]
missing = [f for f in NEEDED if not (FT / f).exists()]
if missing:
    print("pick:", ", ".join(missing))
    up = files.upload()
    for name in list(up):                      # the browser may rename a re-upload, e.g. "chips (1).zip"
        for want in NEEDED:
            stem, ext = os.path.splitext(want)
            if name != want and name.startswith(stem) and name.endswith(ext):
                (FT / name).rename(FT / want); print(f"renamed {name} -> {want}")

missing = [f for f in NEEDED if not (FT / f).exists()]
assert not missing, f"still missing: {missing}: run this cell again and pick them"

with zipfile.ZipFile(FT / "chips.zip") as z:
    bad = z.testzip()
    assert bad is None, f"chips.zip is damaged ({bad}): upload it again"
    z.extractall(FT)
n_train = sum(1 for _ in open(FT / "cover_train.jsonl", encoding="utf-8"))
n_hold = sum(1 for _ in open(FT / "cover_holdout.jsonl", encoding="utf-8"))
n_img = sum(1 for p in FT.rglob("*") if p.suffix.lower() in (".png", ".jpg", ".jpeg"))
print(f"train {n_train} boxes, held-out {n_hold} boxes, {n_img} images")
assert n_train == 327 and n_hold == 80, "unexpected counts: check you uploaded the latest files from data/finetune"

## 4. Baseline, training, evaluation

Three steps in one cell:
1. **Baseline:** the untrained model on the 80 held-out boxes.
2. **Training:** the LoRA adapter, on the 327 training boxes. It prints the loss every 50 boxes.
3. **Evaluation:** the trained adapter on the same 80 boxes.

Each step prints a block of scores, for example `"main_cover_accuracy"` and `"derived_change"`.
The cell stops with a clear message if any step fails.

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path
FT = Path("/content/ft"); OUT = Path("/content/evichange_ft")
os.chdir("/content/EviChange")

def run(*args):
    cmd = [sys.executable, "scripts/finetune_train.py", *args]
    print("\n$", " ".join(cmd[1:]), flush=True)
    p = subprocess.run(cmd)
    if p.returncode != 0:
        raise SystemExit(f"step failed ({' '.join(args[:3])}): send P2 a screenshot of the error above")

if not (FT / "predictions_cover_base.jsonl").exists():
    run("eval", "--task", "cover", "--data", str(FT), "--base-only")
else:
    print("baseline already done: skipped")

if not (OUT / "cover" / "adapter_config.json").exists():
    run("train", "--task", "cover", "--data", str(FT), "--out", str(OUT))
assert (OUT / "cover" / "adapter_config.json").exists(), "training finished without saving an adapter"

run("eval", "--task", "cover", "--data", str(FT), "--adapter", str(OUT / "cover"))
print("\nall done: run cell 5 to download the results")

## 5. Download the results

This downloads `ft_results.zip` (a few hundred MB). **Send it to P2.**

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path
FT = Path("/content/ft"); OUT = Path("/content/evichange_ft")
from google.colab import files
res = Path("/content/ft_results"); shutil.rmtree(res, ignore_errors=True); res.mkdir()
shutil.copytree(OUT, res / "evichange_ft")
for p in FT.glob("predictions_cover_*.jsonl"):
    shutil.copy(p, res / p.name)
shutil.make_archive("/content/ft_results", "zip", res)
print(sorted(str(p.relative_to(res)) for p in res.rglob("*") if p.is_file()))
files.download("/content/ft_results.zip")